# Data Sourcing and Database Integration

## 1. Web Scraping Michelin


In [ ]:
import os

#Type in the directory you wish to store the data
BASE_DIR = "."

#Make sure that the original yelp data is in the same folder that also contains the folder called: "data/processed"
DATA_DIR = os.path.join(BASE_DIR, "data", "raw", "yelp_json")

#Define the directory that stores all the output data
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "processed")

#Input file, the below line of code serve to import the data for filtering out postal code before requesting census API
INPUT_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
import time

URL = 'https://guide.michelin.com/us/en/pennsylvania/philadelphia_2942787/restaurants?sort=distance'
# OUTPUT_FILE = os.path.join(OUTPUT_DIR, "michelin_philadelphia.csv")

#The below line of code define the path that the michelin data will be saved when done scraping
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "michelin_philadelphia.csv")

def scrape_michelin():
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(URL, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')

    cards = soup.select('.js-restaurant__list_item')

    restaurants = []
    for card in cards:
        title_tag = card.select_one('.card__menu-content--title a')
        if not title_tag:
            continue

        name = title_tag.text.strip()
        link = title_tag.get('href', '')
        if link.startswith('/'):
            link = 'https://guide.michelin.com' + link

        lat = card.get('data-lat', '')
        lng = card.get('data-lng', '')
        distinction = card.get('data-map-pin-name', '')

        scores = card.select('.card__menu-footer--score')
        location = scores[0].text.strip() if len(scores) > 0 else ''
        price_and_cuisine = scores[1].text.strip() if len(scores) > 1 else ''

        price_and_cuisine = re.sub(r'\s+', ' ', price_and_cuisine)
        parts = []
        if '·' in price_and_cuisine:
            parts = [p.strip() for p in price_and_cuisine.split('·')]
        elif '•' in price_and_cuisine:
            parts = [p.strip() for p in price_and_cuisine.split('•')]
        else:
            parts = [price_and_cuisine]

        price = parts[0] if len(parts) > 0 else ''
        cuisine = parts[1] if len(parts) > 1 else ''

        if "Philadelphia" not in location:
            continue

        #extract zip code from each resturant detail page since they are not shown on the main page
        #so we need to do another request for each resturant to enter their own detail page

        full_address = ''
        single_zip_code = ''
        zip_regex = r'\d{5}'


        response_3 = requests.get(link, headers=headers)
        response_3.raise_for_status()
        response_3_soup = BeautifulSoup(response_3.content, 'html.parser')

        address_tags = response_3_soup.select('.data-sheet__block')

        for address_tag in address_tags:
            address_tag_info = address_tag.get_text(" ", strip=True)
            #since the zip code is contained in the full address block, we need to first extract the full address and use regex to match the zip code
            if "USA" in address_tag_info:
                full_address = address_tag_info

                zip_match = re.search(zip_regex, address_tag_info)

                if zip_match:
                    single_zip_code = zip_match.group()

        time.sleep(1)

        restaurants.append({
            'name': name,
            'location': location,
            'price': price,
            'cuisine': cuisine,
            'distinction': distinction,
            'latitude': lat,
            'longitude': lng,
            'url': link,
            'postal_code': single_zip_code
        })

    print(restaurants)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    df = pd.DataFrame(restaurants)
    df.drop_duplicates(subset=['name', 'url'], inplace=True)
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Successfully saved {len(df)} restaurants to {OUTPUT_FILE}")
    return df.head()


scrape_michelin()

## 2. Yelp JSON to CSV


In [ ]:
import json
import pandas as pd
import os
from datetime import datetime
from sqlalchemy import create_engine

# Configuration

#The below line of code define the path for the original yelp json data
BUSINESS_FILE = os.path.join(DATA_DIR, "yelp_academic_dataset_business.json")

def process_business_data():
    print("Loading business data...")
    businesses = []
    with open(BUSINESS_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            businesses.append(json.loads(line))

    df_bus = pd.DataFrame(businesses)

    # Filter for Philadelphia Restaurants
    df_philly = df_bus[df_bus['city'].str.lower() == 'philadelphia'].copy()

    def is_restaurant(cats):
        if not cats: return False
        cats_lower = cats.lower()
        targets = ['restaurant', 'food', 'coffee', 'tea', 'bakery', 'cafe', 'bistro', 'delis', 'diners']
        return any(t in cats_lower for t in targets)

    df_philly = df_philly[df_philly['categories'].apply(is_restaurant)].copy()

    # Flatten basic attributes
    df_philly['price_range'] = df_philly['attributes'].apply(lambda x: x.get('RestaurantsPriceRange2') if x else None)

    cols_to_keep = ['business_id', 'name', 'address', 'city', 'state', 'postal_code',
                    'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'price_range']

    df_final = df_philly[cols_to_keep]

    #Turn the scraping data to csv and store them in the path directory we define at the very first cell
    df_final.to_csv(os.path.join(OUTPUT_DIR, "restaurants_philadelphia.csv"), index=False)

    # Import to MySQL
    # engine = create_engine(DB_CONFIG)
    # df_final.to_sql('restaurants', con=engine, if_exists='replace', index=False)

    print(f"Processed {len(df_final)} restaurants.")
    return df_final.head()

process_business_data()

## 3. Census API Lookup


In [ ]:
import pandas as pd
import requests
import json
# import os

#Define the path for the yelp restaurant date that we want to import
restaurants_file = os.path.join(INPUT_DIR, "restaurants_philadelphia.csv")

#Define the path that store the census API when done requesting
output_file = os.path.join(OUTPUT_DIR, "philly_census_data_2.csv")

res_file = pd.read_csv(restaurants_file)

#Turn the value in the column called "postal_code" to string
series_postal_code = res_file["postal_code"].astype(str)

#In the original column, values are shown as "19020.0", so we need to separated the ".0"
string_postal_code = series_postal_code.str.split(".")

#string_postal_code return something like ["19020", "0"], we only need the first([0]) element
updated_postal_code = string_postal_code.str[0]

#Use ".unique()" to filter out duplicated entry
unique_zips = updated_postal_code.unique()

unique_zips = [z for z in unique_zips if z != 'nan' and len(z) == 5]

print(unique_zips)

print(len(unique_zips))

url = "https://api.census.gov/data/2023/acs/acs5"
census_results = []

for z in unique_zips:
    #The reference for CENSUS API extraction  is as follow
    #"https://api.census.gov/data/2023/acs/acs5/examples.html",
    #Spefically, under Column of "Geography level" with value of 860
    #The representation of each variable meaning is reference from "https://api.census.gov/data/2023/acs/acs5/variables.html"
    #For example, "B01003_001E" stands for Total popluation.
    params = {
        "get": "B01003_001E,B19013_001E,B17001_002E,B15003_022E,B01002_001E",
        "for": f"zip code tabulation area:{z}"
    }

    #The requests will automatically generate the corresponding url, for example, for "B01003_001E" and "B19013_001E" with
    #zip of 19107, request will generates the url of "https://api.census.gov/data/2023/acs/acs5?get=B01003_001E,B19013_001E&for=zip code tabulation area:19107"
    response = requests.get(url, params=params)

    #census_data = response.json()
    #I tried using .json() without any restriction to extract corresponding information needed, but there's a
    #status code of 204, basically means that there are no return content. So we need to skip the zip that does
    # not contain information we need.
    if response.status_code == 200:
        census_data = response.json()
        census_results.append(census_data[1])

#Construct dataframe for the extracted data
census_df = pd.DataFrame(
    census_results,
    columns=["population", "median_income", "poverty_count", "bachelor_degree", "median_age", "postal_code"]
)

census_df.to_csv(output_file, index=False)

census_df.head()

## 4. Walkscore API Code


In [ ]:
import pandas as pd
import requests
import time

API_KEY = "YOUR_WALKSCORE_API_KEY"

restaurants_file = os.path.join(INPUT_DIR, "restaurants_philadelphia.csv")
output_file = os.path.join(OUTPUT_DIR, "walkscores_yelp_sample.csv")

# Yelp sample only for notebook demo
df = pd.read_csv(restaurants_file).head(10)


def get_walkscore(address, lat, lon):
    url = "https://api.walkscore.com/score"
    params = {
        "format": "json",
        "address": address,
        "lat": lat,
        "lon": lon,
        "transit": 1,
        "bike": 1,
        "wsapikey": API_KEY
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        return resp.json()
    except:
        return {"status": "request_failed"}

results = []
print("Fetching Walkscores for sample Yelp restaurants...")

for i, row in df.iterrows():
    score_data = get_walkscore(row["address"], row["latitude"], row["longitude"])
    results.append({
        "source": "yelp_sample",
        "business_id": row["business_id"],
        "name": row["name"],
        "walkscore": score_data.get("walkscore"),
        "transit_score": score_data.get("transit", {}).get("score"),
        "bike_score": score_data.get("bike", {}).get("score")
    })
    time.sleep(0.1)

walkscore_df = pd.DataFrame(results)
walkscore_df.to_csv("data/processed/walkscores_yelp_sample.csv", index=False)
walkscore_df.to_csv(output_file, index=False)


print("Walkscore fetch complete.")
display(walkscore_df.head())

In [ ]:
import pandas as pd
import requests
import time

API_KEY = "YOUR_WALKSCORE_API_KEY"

input_file = os.path.join(INPUT_DIR, "michelin_philadelphia.csv")
partial_output = os.path.join(OUTPUT_DIR, "michelin_philadelphia_with_walkscore_partial.csv")
final_output = os.path.join(OUTPUT_DIR, "michelin_philadelphia_with_walkscore_final.csv")

df = pd.read_csv(input_file)

# Use location directly as address
df["full_address"] = df["location"].fillna("").astype(str)

# Create new columns to store Walk Score results
df["walkscore"] = None
df["walkscore_description"] = None
df["transit_score"] = None
df["transit_description"] = None
df["bike_score"] = None
df["bike_description"] = None
df["walkscore_status"] = None
df["ws_link"] = None

# Function to call Walk Score API
def get_walkscore(address, lat, lon):
    url = "https://api.walkscore.com/score"
    params = {
        "format": "json",
        "address": address,
        "lat": lat,
        "lon": lon,
        "transit": 1,
        "bike": 1,
        "wsapikey": API_KEY
    }

    try:
        resp = requests.get(url, params=params, timeout=10)
        return resp.json()
    except:
        return {"status": "request_failed"}

# Loop through every Michelin restaurant
for i, row in df.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    address = row["full_address"]

    result = get_walkscore(address, lat, lon)

    df.at[i, "walkscore_status"] = result.get("status")
    df.at[i, "ws_link"] = result.get("ws_link")

    if result.get("status") == 1:
        df.at[i, "walkscore"] = result.get("walkscore")
        df.at[i, "walkscore_description"] = result.get("description")

        transit = result.get("transit", {})
        df.at[i, "transit_score"] = transit.get("score")
        df.at[i, "transit_description"] = transit.get("description")

        bike = result.get("bike", {})
        df.at[i, "bike_score"] = bike.get("score")
        df.at[i, "bike_description"] = bike.get("description")

    time.sleep(0.1)

    if i % 10 == 0:
        df.to_csv(partial_output, index=False)
        print(f"Saved progress at row {i}")

# Save final result
df.to_csv(final_output, index=False)
print("Finished.")

## 5. Data Cleaning


In [ ]:
import pandas as pd
import numpy as np

# yelp = pd.read_csv("restaurants_philadelphia_with_walkscore_final.csv")
# michelin = pd.read_csv("michelin_philadelphia_with_walkscore_final.csv")
# census = pd.read_csv("philly_census_data.csv")

#Define the path of the input file that we want to clean
yelp_file = os.path.join(INPUT_DIR, "restaurants_philadelphia_with_walkscore_final.csv")
michelin_file = os.path.join(INPUT_DIR, "michelin_philadelphia_with_walkscore_final.csv")
census_file = os.path.join(INPUT_DIR, "philly_census_data_2.csv")

#Define output files
yelp_output = os.path.join(OUTPUT_DIR, "yelp_clean.csv")
michelin_output = os.path.join(OUTPUT_DIR, "michelin_clean.csv")
census_output = os.path.join(OUTPUT_DIR, "census_clean.csv")
match_output = os.path.join(OUTPUT_DIR, "michelin_yelp_match.csv")

#Read files
yelp = pd.read_csv(yelp_file)
michelin = pd.read_csv(michelin_file)
census = pd.read_csv(census_file)

#Clean column names
# yelp.columns = yelp.columns.str.strip().str.lower().str.replace(" ", "_")
# michelin.columns = michelin.columns.str.strip().str.lower().str.replace(" ", "_")
# census.columns = census.columns.str.strip().str.lower().str.replace(" ", "_")

# Clean Yelp column names
#For every columns in the yelp data, we remove all the whitespace befre and after each column name,
#change all letters to lowercase and replace whitespace with underscore.
#Simliar procedure for the rest of the table.
new_columns = []
for col in yelp.columns:
    col = col.strip()
    col = col.lower()
    col = col.replace(" ", "_")
    new_columns.append(col)
yelp.columns = new_columns

# Clean Michelin column names
new_columns = []
for col in michelin.columns:
    col = col.strip()
    col = col.lower()
    col = col.replace(" ", "_")
    new_columns.append(col)
michelin.columns = new_columns

# Clean Census column names
new_columns = []
for col in census.columns:
    col = col.strip()
    col = col.lower()
    col = col.replace(" ", "_")
    new_columns.append(col)
census.columns = new_columns

#Clean postal_code
# yelp["postal_code"] = yelp["postal_code"].astype(str).str.replace(".0", "", regex=False)
# michelin["postal_code"] = michelin["postal_code"].astype(str).str.replace(".0", "", regex=False)
# census["postal_code"] = census["postal_code"].astype(str).str.replace(".0", "", regex=False)

#Clean Yelp postal_code
#for the value in postal_code filed of yelp data, we first turn the type of data to string(in case the original data stroed float)
#Then we delete the ".0", since in the original filed, postal code is something like "19107.0"
#The rest of the table follows the similar procedure
clean_postal = []
for val in yelp["postal_code"]:
    if pd.isna(val):
        clean_postal.append(None)
    else:
        val = str(val)
        val = val.replace(".0", "")
        clean_postal.append(val)
yelp["postal_code"] = clean_postal

#Clean Michelin postal_code
clean_postal = []
for val in michelin["postal_code"]:
    if pd.isna(val):
        clean_postal.append(None)
    else:
        val = str(val)
        val = val.replace(".0", "")
        clean_postal.append(val)
michelin["postal_code"] = clean_postal

#Clean Census postal_code
clean_postal = []
for val in census["postal_code"]:
    if pd.isna(val):
        clean_postal.append(None)
    else:
        val = str(val)
        val = val.replace(".0", "")
        clean_postal.append(val)
census["postal_code"] = clean_postal

sentinel_values = [-666666666, -666666666.0, "-666666666", "-666666666.0"]
census = census.replace(sentinel_values, np.nan)

#Add new primary keys
yelp.insert(0, "restaurant_id", range(1, len(yelp) + 1))
michelin.insert(0, "michelin_id", range(1, len(michelin) + 1))

#Convert Michelin price to number
#Turn every  price entry to string
michelin_price_str = michelin["price"].astype(str)

#Count how many "$" for each entry, and turn them into number
price_level_list = []
for val in michelin_price_str:
    count = val.count("$")
    price_level_list.append(count)

michelin["price_level"] = price_level_list

#Clean restaurant names for yelp
#Similarly, turn value of each entry to string
yelp_name_str = yelp["name"].astype(str)

#Turn all letters for the name to lowercase
yelp_name_lower = []
for val in yelp_name_str:
    val = val.lower()
    yelp_name_lower.append(val)

#Take away the space around each restaurant name
yelp_name_clean = []
for val in yelp_name_lower:
    val = val.strip()
    yelp_name_clean.append(val)

yelp["name_clean"] = yelp_name_clean

#Clean restaurant name for michelin
#Follows the similar procedure for cleaning yelp restaurant names
michelin_name_str = michelin["name"].astype(str)

michelin_name_lower = []
for val in michelin_name_str:
    val = val.lower()
    michelin_name_lower.append(val)

michelin_name_clean = []
for val in michelin_name_lower:
    val = val.strip()
    michelin_name_clean.append(val)

michelin["name_clean"] = michelin_name_clean


#Check for missing value in each table, printing them out allows us to
#trace back to the original table to see the type of value in missing column
#and come up with corresponding method to handle them.
# print("\nMissing values in Yelp:")
# print(yelp.isna().sum())

# print("\nMissing values in Michelin:")
# print(michelin.isna().sum())

# print("\nMissing values in Census:")
# print(census.isna().sum())

#Do simple name matching
match_list = []

for i in range(len(michelin)):
    michelin_id = michelin.loc[i, "michelin_id"]
    michelin_name = michelin.loc[i, "name"]
    michelin_name_clean = michelin.loc[i, "name_clean"]

    found = yelp[yelp["name_clean"] == michelin_name_clean]

    if len(found) > 0:
        restaurant_id = found.iloc[0]["restaurant_id"]
        business_id = found.iloc[0]["business_id"]
        yelp_name = found.iloc[0]["name"]
        match_status = "matched"
    else:
        restaurant_id = np.nan
        business_id = np.nan
        yelp_name = np.nan
        match_status = "unmatched"

    match_list.append({
        "michelin_id": michelin_id,
        "michelin_name": michelin_name,
        "restaurant_id": restaurant_id,
        "business_id": business_id,
        "yelp_name": yelp_name,
        "match_status": match_status
    })

match = pd.DataFrame(match_list)

#Clean match restaurant_id for SQL
match["restaurant_id"] = match["restaurant_id"].apply(
    lambda x: int(x) if pd.notna(x) else None
)

#Add is_michelin to Yelp
matched_ids = match["restaurant_id"].dropna().tolist()
yelp["is_michelin"] = yelp["restaurant_id"].isin(matched_ids).astype(int)

#Drop helper columns
yelp_clean = yelp.drop(columns=["name_clean"])
michelin_clean = michelin.drop(columns=["name_clean"])

#Convert missing values to None so they can become NULL in SQL import
yelp_clean = yelp_clean.where(pd.notnull(yelp_clean), None)
michelin_clean = michelin_clean.where(pd.notnull(michelin_clean), None)
census = census.where(pd.notnull(census), None)
match = match.where(pd.notnull(match), None)

#Save files
yelp_clean.to_csv(yelp_output, index=False)
michelin_clean.to_csv(michelin_output, index=False)
census.to_csv(census_output, index=False)
match.to_csv(match_output, index=False)

#Print summary
print("Yelp rows:", len(yelp_clean))
print("Michelin rows:", len(michelin_clean))
print("Census rows:", len(census))
print("Matched Michelin restaurants:", (match["match_status"] == "matched").sum())
print("Unmatched Michelin restaurants:", (match["match_status"] == "unmatched").sum())

print(match[["michelin_name", "yelp_name", "match_status"]])

In [ ]:
census_df = pd.read_csv("data/processed/philly_census_data_2.csv")
yelp_df = pd.read_csv("data/processed/restaurants_philadelphia.csv")

# Ensure ZIP codes are strings
census_zips = set(census_df["postal_code"].astype(str))
yelp_zips = set(yelp_df["postal_code"].astype(str))

# Find ZIP codes present in Yelp but missing from Census
missing_zips = yelp_zips - census_zips

print("Missing zip codes in census:")
print(missing_zips)
print("Count:", len(missing_zips))

## 6. Inserting data to mysql


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# connect to MySQL database
#Input account and password here
# engine = create_engine()

# read csv files
census = pd.read_csv("census_clean.csv")
yelp = pd.read_csv("yelp_clean.csv")
michelin = pd.read_csv("michelin_clean.csv")
match = pd.read_csv("michelin_yelp_match.csv")

# replace NaN with None (so SQL can read as NULL)
census = census.where(pd.notnull(census), None)
yelp = yelp.where(pd.notnull(yelp), None)
michelin = michelin.where(pd.notnull(michelin), None)
match = match.where(pd.notnull(match), None)

# keep only columns that match SQL tables
census = census[
    ["postal_code", "population", "median_income", "poverty_count", "bachelor_degree", "median_age"]
]

yelp = yelp[
    ["restaurant_id", "business_id", "name", "address", "city", "state", "postal_code",
     "latitude", "longitude", "stars", "review_count", "price_range",
     "is_open", "is_michelin", "walkscore", "transit_score", "bike_score"]
]

michelin = michelin[
    ["michelin_id", "name", "postal_code", "latitude", "longitude",
     "price_level", "cuisine", "distinction", "walkscore", "transit_score", "bike_score"]
]

match = match[
    ["michelin_id", "restaurant_id", "business_id", "yelp_name", "match_status"]
]

# insert data into MySQL (append = add rows)
census.to_sql("census_data", con=engine, if_exists="append", index=False)
yelp.to_sql("yelp_restaurants", con=engine, if_exists="append", index=False)
michelin.to_sql("michelin_restaurants", con=engine, if_exists="append", index=False)
match.to_sql("michelin_yelp_match", con=engine, if_exists="append", index=False)

print("All data imported successfully.")